# Contraction Hierarchy Bidirectional Search Demo

This notebook reproduces the logic from the Python prototype used to validate the C++ query engine. It loads the shortcut and edge metadata tables, rebuilds the adjacency lists, and exercises the hierarchy-aware bidirectional Dijkstra implementation.

## Load Edge Metadata

Read the precomputed edge CSV to recover incoming cells and hierarchy resolution used as constraints during search.

In [23]:
import pandas as pd

# Update this path if you want to work with a different edge metadata snapshot.
edges_PATH = "../../spark-shortest-path/data/Burnaby_driving_simplified_edges_with_h3.csv"

# Load the edge table and retain it with the edge id as index for fast lookups.
edges = pd.read_csv(edges_PATH)
edges_df = edges.set_index('id')
print(f"Loaded {len(edges):,} all edges")
edges.head()

Loaded 34,965 all edges


,length,maxspeed,geometry,highway,cost,incoming_cell,outgoing_cell,lca_res,id
0,98.503,30.0,LINESTRING (-122.92154693603516 49.27764129638...,tertiary_link,11.82036,644733695069033029,644733695069283890,8,0
1,53.933,30.0,LINESTRING (-122.92154693603516 49.27764129638...,tertiary,6.47196,644733695069345412,644733695069283890,9,1
2,684.050,50.0,LINESTRING (-122.92154693603516 49.27764129638...,tertiary,49.25160,644733695063774494,644733695069283890,7,2
3,35.488,30.0,LINESTRING (-122.92096710205078 49.27793121337...,tertiary,4.25856,644733695069337920,644733695069345412,10,3
4,53.933,30.0,LINESTRING (-122.92096710205078 49.27793121337...,tertiary,6.47196,644733695069283890,644733695069345412,9,4


## Load Shortcut Parquet

Read the precomputed shortcut catalog, enforce numeric types, and inspect the first few rows.

In [24]:
PARQUET_PATH = "../../spark-shortest-path/output/Burnaby_shortcuts_final"

# Load shortcuts and normalise schema for downstream computations.
df = pd.read_parquet(PARQUET_PATH)

# Ensure deterministic types for the routing algorithm.
df['incoming_edge'] = df['incoming_edge'].astype(int)
df['outgoing_edge'] = df['outgoing_edge'].astype(int)
df['via_edge'] = df['via_edge'].fillna(0).astype(int)
df['inside'] = df['inside'].astype(int)
df['cost'] = df['cost'].astype(float)

print(f"Loaded {len(df):,} shortcut edges")
df.head()

Loaded 2,273,221 shortcut edges


,incoming_edge,outgoing_edge,cost,via_edge,inside,cell
0,19,1128,17.858967,1128,0,608704929887944703
1,30,16904,61.799682,26838,0,608704897809907711
2,46,7747,39.198300,7747,1,608704929871167487
3,46,9134,34.085300,9134,0,608704929871167487
4,57,9152,57.419547,9152,0,608704929871167487


## Build Forward and Backward Adjacency Lists

Convert the shortcut table into adjacency dictionaries that the bidirectional search consumes.

In [25]:
import heapq

# Build Adjacency Lists
# fwd_adj: u -> [(v, cost, inside), ...]
# bwd_adj: v -> [(u, cost, inside), ...]

print("Building Graph...")
fwd_adj = {}
bwd_adj = {}

# We can iterate over the dataframe or convert to records for speed
records = df[['incoming_edge', 'outgoing_edge', 'cost', 'inside','cell']].to_dict('records')

for row in records:
    u = row['incoming_edge']
    v = row['outgoing_edge']
    c = row['cost']
    inside = row['inside']
    cell = row['cell']
    
    if u not in fwd_adj: fwd_adj[u] = []
    fwd_adj[u].append((v, c, cell, inside))
    
    if v not in bwd_adj: bwd_adj[v] = []
    bwd_adj[v].append((u, c, cell, inside))

shortcut_lookup = df.set_index(['incoming_edge', 'outgoing_edge'])

print(f"Graph built. Nodes with outgoing edges: {len(fwd_adj)}, Nodes with incoming edges: {len(bwd_adj)}")


Building Graph...
Graph built. Nodes with outgoing edges: 34965, Nodes with incoming edges: 34965
Graph built. Nodes with outgoing edges: 34965, Nodes with incoming edges: 34965


## H3 Utilities

Utility helpers to compute ancestor cells and verify that shortcuts stay within the allowable hierarchy.

## Bidirectional Dijkstra

Implementation of the hierarchy-aware search that processes upward shortcuts forward and downward (plus lateral-at-top) shortcuts backward.

In [26]:
import h3


def _find_ancestor_impl(cell: int, res: int) -> int:
    """Return the ancestor of ``cell`` at resolution ``res`` (or 0 if invalid)."""
    if cell == 0 or res < 0:
        return 0
    if res > h3.get_resolution(h3.int_to_str(cell)):
        return cell
    return h3.str_to_int(h3.cell_to_parent(h3.int_to_str(cell), res))

def find_lca(cell1: int, cell2: int) -> int:
    """Compute the lowest common ancestor between two H3 cells."""
    if cell1 == 0 or cell2 == 0:
        return 0
    cell1_res = h3.get_resolution(h3.int_to_str(cell1))
    cell2_res = h3.get_resolution(h3.int_to_str(cell2))
    lca_res = min(cell1_res, cell2_res)
    while lca_res >= 0:
        if h3.cell_to_parent(h3.int_to_str(cell1), lca_res) == h3.cell_to_parent(h3.int_to_str(cell2), lca_res):
            return h3.str_to_int(h3.cell_to_parent(h3.int_to_str(cell1), lca_res))
        lca_res -= 1
    return 0

def high_cell(source_edge_id: int, destination_edge_id: int, edges_df: pd.DataFrame) -> int:
    """Return the highest common ancestor cell for the two edge endpoints."""
    s_cell = edges_df.loc[source_edge_id]["incoming_cell"]
    s_res = edges_df.loc[source_edge_id]["lca_res"]
    d_cell = edges_df.loc[destination_edge_id]["incoming_cell"]
    d_res = edges_df.loc[destination_edge_id]["lca_res"]
    source_cell = _find_ancestor_impl(s_cell, res=s_res)
    destination_cell = _find_ancestor_impl(d_cell, res=d_res)
    return find_lca(source_cell, destination_cell)

def high_cell_resolution(highcell: int) -> int:
    if highcell == 0:
        return -1
    return h3.get_resolution(h3.int_to_str(highcell))

def parent_check(child_cell: int, parent_cell: int, parent_res: int) -> bool:
    """Verify that ``child_cell`` lies within ``parent_cell`` at ``parent_res``."""
    if parent_cell == 0:
        return True
    if child_cell == 0:
        return False
    child_res = h3.get_resolution(h3.int_to_str(child_cell))
    if parent_res > child_res:
        return False
    derived_parent = _find_ancestor_impl(child_cell, parent_res)
    return derived_parent == parent_cell

In [27]:
def bidijkstra(source: int, target: int, highcell: int, highcellres: int, target_cost: float = 0.0):
    """Bidirectional Dijkstra constrained by the CH hierarchy.
    
    Parameters
    ----------
    source : int
        Source edge id.
    target : int
        Target edge id.
    highcell : int
        Highest common ancestor H3 cell.
    highcellres : int
        Resolution of the highest common ancestor cell.
    target_cost : float, optional
        Cost/weight of the destination edge to include in the total path cost.
        Default is 0.0 (not included).
    
    Returns
    -------
    tuple[float, list[int]]
        (total_cost, path) where total_cost includes target_cost if provided.
    """
    if source == target:
        return target_cost, [source]

    dist_fwd = {source: 0.0}
    dist_bwd = {target: 0.0}
    prev_fwd = {source: None}
    prev_bwd = {target: None}

    pq_fwd = [(0.0, source)]
    pq_bwd = [(0.0, target)]

    best_cost = float("inf")
    meeting_node = None

    while pq_fwd or pq_bwd:
        if pq_fwd:
            d, u = heapq.heappop(pq_fwd)
            if d > dist_fwd.get(u, float("inf")):
                continue

            if u in dist_bwd:
                total = d + dist_bwd[u]
                if total < best_cost:
                    best_cost = total
                    meeting_node = u

            if d >= best_cost:
                continue

            for v, cost, cell, inside in fwd_adj.get(u, []):
                if inside != 1 or not parent_check(cell, highcell, highcellres):
                    continue
                new_dist = d + cost
                if new_dist < dist_fwd.get(v, float("inf")):
                    dist_fwd[v] = new_dist
                    prev_fwd[v] = u
                    heapq.heappush(pq_fwd, (new_dist, v))

                    if v in dist_bwd:
                        total = new_dist + dist_bwd[v]
                        if total < best_cost:
                            best_cost = total
                            meeting_node = v

        if pq_bwd:
            d, v = heapq.heappop(pq_bwd)
            if d > dist_bwd.get(v, float("inf")):
                continue

            if v in dist_fwd:
                total = d + dist_fwd[v]
                if total < best_cost:
                    best_cost = total
                    meeting_node = v

            if d >= best_cost:
                continue

            for u, cost, cell, inside in bwd_adj.get(v, []):
                allow_lateral = highcell != 0 and inside == 0 and cell == highcell
                if inside not in (-1, 0) or (inside == 0 and not allow_lateral):
                    continue
                if not parent_check(cell, highcell, highcellres):
                    continue
                new_dist = d + cost
                if new_dist < dist_bwd.get(u, float("inf")):
                    dist_bwd[u] = new_dist
                    prev_bwd[u] = v
                    heapq.heappush(pq_bwd, (new_dist, u))

                    if u in dist_fwd:
                        total = new_dist + dist_fwd[u]
                        if total < best_cost:
                            best_cost = total
                            meeting_node = u

        if best_cost < float("inf"):
            if (
                pq_fwd
                and pq_fwd[0][0] >= best_cost
                and pq_bwd
                and pq_bwd[0][0] >= best_cost
            ):
                break

    if meeting_node is None:
        return -1, []

    forward_path = []
    node = meeting_node
    while node is not None:
        forward_path.append(node)
        node = prev_fwd.get(node)
    forward_path.reverse()

    backward_path = []
    node = prev_bwd.get(meeting_node)
    while node is not None:
        backward_path.append(node)
        node = prev_bwd.get(node)

    path = forward_path + backward_path
    # Add target_cost to the final path cost
    final_cost = best_cost + target_cost
    return final_cost, path

In [28]:
def expand_shortcut_path(shortcut_path, shortcuts_df):
    """Expand a list of shortcut edge ids into the underlying base edge ids.

    The path is a sequence of edges where consecutive pairs (u, v) represent
    shortcuts. Each shortcut may have a via_edge that itself needs expansion.

    Parameters
    ----------
    shortcut_path : list[int]
        Ordered sequence of edge ids returned by the bidirectional search.
    shortcuts_df : pandas.DataFrame
        Shortcut table indexed by ``(incoming_edge, outgoing_edge)`` with column ``via_edge``.

    Returns
    -------
    list[int]
        Sequence of base edge ids after recursively expanding all shortcuts.
    """
    if not shortcut_path:
        return []
    if len(shortcut_path) == 1:
        return [int(shortcut_path[0])]
    
    def expand_pair(u, v, visited=None):
        """Recursively expand a shortcut (u -> v) into base edges."""
        if visited is None:
            visited = set()
        pair = (u, v)
        if pair in visited:
            return [u, v]  # Cycle detected
        visited.add(pair)
        
        try:
            row = shortcuts_df.loc[(u, v)]
        except KeyError:
            # No shortcut entry, these are consecutive base edges
            return [u, v]
        
        if isinstance(row, pd.DataFrame):
            row = row.iloc[0]
        
        via = row.get('via_edge', 0)
        via = 0 if pd.isna(via) else int(via)
        
        if via == 0:
            # Base edge (no intermediate), just return the pair
            return [u, v]
        
        # Recursively expand: u -> via and via -> v
        left = expand_pair(u, via, visited.copy())
        right = expand_pair(via, v, visited.copy())
        
        # Merge, avoiding duplicate at the junction
        if right and right[0] == left[-1]:
            return left + right[1:]
        return left + right
    
    # Expand each consecutive pair and merge
    base_edges = []
    for u, v in zip(shortcut_path, shortcut_path[1:]):
        expanded = expand_pair(int(u), int(v))
        for e in expanded:
            if not base_edges or base_edges[-1] != e:
                base_edges.append(e)
    
    return base_edges

## Expand Shortcuts to Base Edges

Translate the CH-level path into the underlying edge ids using the `via_edge` mapping.

In [32]:
import random
from time import perf_counter

# Sample one pair (or swap to random sampling) and run the bidirectional search.
all_nodes = list(set(df['incoming_edge'].unique()) | set(df['outgoing_edge'].unique()))
source = random.choice(all_nodes)  # for stochastic testing
target = random.choice(all_nodes)  # for stochastic testing

print(f"Query: {source} -> {target}")
highcell = high_cell(source, target, edges_df)
highcellres = high_cell_resolution(highcell)

# Get the cost of the destination edge from the edges table
# Adjust 'length' to the actual column name for edge weight/cost in your edges_df
target_cost = edges_df.loc[target].get('length', 0.0) if target in edges_df.index else 0.0

t_start = perf_counter()

distance, path = bidijkstra(source, target, highcell, highcellres, target_cost=target_cost)
elapsed = perf_counter() - t_start

if distance == -1:
    print("No path found.")
else:
    print(f"Distance (including destination edge): {distance}")
    print(f"Destination edge cost: {target_cost}")
    print(f"Path length: {len(path)} nodes")
    base_path = expand_shortcut_path(path, shortcut_lookup)
    print(f"Expanded base edge path: {base_path}")
    print(f"Shortcut path: {path}")

print(f"Runtime: {elapsed * 1000:.2f} ms")

Query: 4547 -> 24023
Distance (including destination edge): 146.61783666666668
Destination edge cost: 9.479
Path length: 7 nodes
Expanded base edge path: [4547, 4550, 25004, 25910, 30827, 3929, 24023]
Shortcut path: [np.int64(4547), 4550, 25004, 25910, 30827, 3929, np.int64(24023)]
Runtime: 153.07 ms


In [30]:
shortcut_path = path
for current_edge, next_edge in zip(shortcut_path, shortcut_path[1:]):
    print(current_edge, next_edge)
    print(shortcut_lookup.loc[(current_edge, next_edge)].apply(int))
    print("----------------------")

13790 13786
cost                         4
via_edge                 13786
inside                       1
cell        622215728747872256
Name: (13790, 13786), dtype: int64
----------------------
13786 7734
cost                         7
via_edge                  7734
inside                       1
cell        617712129120665600
Name: (13786, 7734), dtype: int64
----------------------
7734 8818
cost                         9
via_edge                  8818
inside                       1
cell        608704929871167488
Name: (7734, 8818), dtype: int64
----------------------
8818 16414
cost                        52
via_edge                   191
inside                       1
cell        599697731186851840
Name: (8818, 16414), dtype: int64
----------------------
16414 8353
cost                         9
via_edge                  8353
inside                      -1
cell        613208497397432320
Name: (16414, 8353), dtype: int64
----------------------


In [31]:
(current_edge, next_edge) = (17742, 24667)

shortcut_lookup.loc[(current_edge, next_edge)].apply(int).reset_index()

,index,17742
,,24667
0,cost,35
1,via_edge,24667
2,inside,1
3,cell,608704929619509248
